# 05. Clustering: particiones (etapa 2)

Aplica tres algoritmos a las tres matrices de distancia del notebook 04 (euclídea, DTW y SBD), para k de 2 a 8. Una configuración principal por algoritmo:

- **Jerárquico** con enlace de Ward.
- **Spectral** con afinidad gaussiana de escala local.
- **PAM** (k-medoides).

Esta etapa sólo genera las particiones y diagnostica sus tamaños. La elección de k (etapa 3) y la estabilidad con semillas (etapa 4) van después.

**Entrada:** `chartmetric_pilotos/distancias_clustering.rds` (notebook 04).

In [ ]:
# PAQUETES

library(data.table)
library(cluster)
library(ggplot2)

## 1. Configuración y carga

In [ ]:
# ============================================================
# CONFIGURACION
# ============================================================

ARCHIVO_DISTANCIAS <- file.path("chartmetric_pilotos", "distancias_clustering.rds")
ARCHIVO_PARTICIONES <- file.path("chartmetric_pilotos", "particiones_clustering.rds")
FIGURES_DIR <- "figuras"

K_CANDIDATOS <- 2:8

# Jerárquico: enlace de Ward sobre distancias (ward.D2 eleva al
# cuadrado las distancias, como corresponde al criterio de Ward)
ENLACE_JERARQUICO <- "ward.D2"

# Spectral: escala local de la afinidad = distancia de cada artista
# a su k-ésimo vecino más cercano; k-means con varios inicios
VECINO_ESCALA_LOCAL <- 7L
INICIOS_KMEANS <- 25L
N_AUTOVALORES_GUARDADOS <- 15L

ETIQUETAS_DISTANCIA <- c(euclidea = "Euclídea", dtw = "DTW", sbd = "SBD")
ETIQUETAS_ALGORITMO <- c(jerarquico = "Jerárquico (Ward)", espectral = "Spectral", pam = "PAM")

# Semillas: misma función y misma semilla primigenia que el notebook 04
SEMILLA_PRIMIGENIA <- 999199L

primos_seis_cifras <- function() {
  limite <- 999999L
  es_primo <- rep(TRUE, limite)
  es_primo[1] <- FALSE
  for (p in 2:floor(sqrt(limite))) {
    if (es_primo[p]) es_primo[seq(p * p, limite, by = p)] <- FALSE
  }
  candidatos <- which(es_primo)
  candidatos[candidatos >= 100000L]
}

PRIMOS_SEIS_CIFRAS <- primos_seis_cifras()

generar_semillas_primas <- function(n, semilla = SEMILLA_PRIMIGENIA) {
  set.seed(semilla)
  orden <- sample(length(PRIMOS_SEIS_CIFRAS))
  PRIMOS_SEIS_CIFRAS[orden[seq_len(n)]]
}

# Control: las primeras semillas coinciden con las del notebook 04
stopifnot(identical(generar_semillas_primas(3), c(742457L, 895957L, 543139L)))

# Semilla del k-means del spectral (la primera de la serie)
SEMILLA_KMEANS_ESPECTRAL <- generar_semillas_primas(1)

SEMILLA_KMEANS_ESPECTRAL

In [ ]:
# ============================================================
# CARGA DE LAS MATRICES DEL NOTEBOOK 04
# ============================================================

stopifnot("Faltan las matrices: correr el notebook 04" = file.exists(ARCHIVO_DISTANCIAS))

distancias <- readRDS(ARCHIVO_DISTANCIAS)
matrices_distancia <- distancias$matrices
artistas <- distancias$artistas
ids_artistas <- as.character(artistas$artist_id)

stopifnot(
  identical(names(matrices_distancia), names(ETIQUETAS_DISTANCIA)),
  all(vapply(matrices_distancia, function(m) identical(rownames(m), ids_artistas), logical(1)))
)

cat(
  "Matrices generadas:", format(distancias$fecha_generacion), "\n",
  "Artistas:", length(ids_artistas), "\n",
  "Distancias:", paste(names(matrices_distancia), collapse = ", "), "\n",
  "Ventana DTW:", distancias$ventana_dtw_meses, "meses\n"
)

## 2. Jerárquico (Ward)

Une en cada paso los dos grupos que menos aumentan la varianza interna. Usa la matriz directamente y es determinístico: no necesita semilla. El dendrograma es el insumo de la lectura de k (etapa 3).

In [ ]:
# ============================================================
# JERARQUICO: un árbol por distancia y cortes en cada k
# ============================================================

arboles_jerarquicos <- lapply(
  matrices_distancia,
  function(m) hclust(as.dist(m), method = ENLACE_JERARQUICO)
)

particiones_jerarquico <- rbindlist(lapply(names(arboles_jerarquicos), function(distancia) {
  cortes <- cutree(arboles_jerarquicos[[distancia]], k = K_CANDIDATOS)
  rbindlist(lapply(K_CANDIDATOS, function(k) data.table(
    artist_id = ids_artistas,
    algoritmo = "jerarquico",
    distancia = distancia,
    k = k,
    cluster = unname(cortes[ids_artistas, as.character(k)])
  )))
}))

In [ ]:
# ============================================================
# FIGURA: dendrogramas de las tres distancias
# ============================================================

COLORES_DISTANCIA <- c(euclidea = "#1982C4", dtw = "#BC4B51", sbd = "#6A4C93")

grafico_dendrogramas <- function() {
  par(mfrow = c(1, 3), bg = "white", mar = c(2, 4.5, 4, 1), oma = c(0, 0, 2.5, 0), las = 1)
  for (distancia in names(arboles_jerarquicos)) {
    plot(
      as.dendrogram(arboles_jerarquicos[[distancia]]),
      leaflab = "none",
      edgePar = list(col = COLORES_DISTANCIA[[distancia]], lwd = 0.6),
      main = ETIQUETAS_DISTANCIA[[distancia]],
      ylab = "Altura de fusión"
    )
  }
  mtext("Clustering jerárquico (Ward) de 1252 trayectorias", side = 3, outer = TRUE, line = 0.5, cex = 1.1)
}

grafico_dendrogramas()

png(file.path(FIGURES_DIR, "clustering_02_dendrogramas.png"), width = 3600, height = 1500, res = 300)
grafico_dendrogramas()
invisible(dev.off())

## 3. Spectral

Convierte cada distancia en afinidad, `exp(-d² / (σᵢ σⱼ))`, con σᵢ igual a la distancia de cada artista a su 7.° vecino (escala local: se adapta a zonas densas y dispersas). Toma los k primeros autovectores de la afinidad normalizada y agrupa con k-means en ese espacio. Así admite cualquier distancia, incluidas DTW y SBD. El k-means usa la semilla derivada de 999199.

In [ ]:
# ============================================================
# SPECTRAL: afinidad, autovectores y k-means por k
# ============================================================

afinidad_escala_local <- function(m, vecino = VECINO_ESCALA_LOCAL) {
  escala <- apply(m, 1, function(fila) sort(fila)[vecino + 1])
  afinidad <- exp(-m^2 / outer(escala, escala))
  diag(afinidad) <- 0
  afinidad
}

# Autovectores de la afinidad normalizada D^(-1/2) A D^(-1/2):
# sus mayores autovalores corresponden a los menores del laplaciano
descomposicion_espectral <- function(m) {
  afinidad <- afinidad_escala_local(m)
  grado <- rowSums(afinidad)
  normalizada <- afinidad / sqrt(outer(grado, grado))
  eigen(normalizada, symmetric = TRUE)
}

descomposiciones <- lapply(matrices_distancia, descomposicion_espectral)

agrupar_espectral <- function(descomposicion, k) {
  incrustacion <- descomposicion$vectors[, seq_len(k), drop = FALSE]
  incrustacion <- incrustacion / sqrt(rowSums(incrustacion^2))
  set.seed(SEMILLA_KMEANS_ESPECTRAL)
  kmeans(incrustacion, centers = k, nstart = INICIOS_KMEANS, iter.max = 100)$cluster
}

particiones_espectral <- rbindlist(lapply(names(descomposiciones), function(distancia) {
  rbindlist(lapply(K_CANDIDATOS, function(k) data.table(
    artist_id = ids_artistas,
    algoritmo = "espectral",
    distancia = distancia,
    k = k,
    cluster = agrupar_espectral(descomposiciones[[distancia]], k)
  )))
}))

# Autovalores guardados para el eigengap de la etapa 3
autovalores_espectral <- rbindlist(lapply(names(descomposiciones), function(distancia) data.table(
  distancia = distancia,
  orden = seq_len(N_AUTOVALORES_GUARDADOS),
  autovalor = descomposiciones[[distancia]]$values[seq_len(N_AUTOVALORES_GUARDADOS)]
)))

# Sanity check: el primer autovalor de la afinidad normalizada es 1
stopifnot(all(abs(autovalores_espectral[orden == 1, autovalor] - 1) < 1e-8))

## 4. PAM (k-medoides)

Elige k artistas reales como centros (medoides) y asigna cada artista al más cercano. Usa la matriz directamente y es determinístico (fases BUILD y SWAP): no necesita semilla. El medoide sirve para contar cada perfil con un caso.

In [ ]:
# ============================================================
# PAM: una corrida por distancia y k
# ============================================================

resultados_pam <- lapply(matrices_distancia, function(m) {
  lapply(setNames(K_CANDIDATOS, K_CANDIDATOS), function(k) pam(as.dist(m), k = k, diss = TRUE))
})

particiones_pam <- rbindlist(lapply(names(resultados_pam), function(distancia) {
  rbindlist(lapply(K_CANDIDATOS, function(k) data.table(
    artist_id = ids_artistas,
    algoritmo = "pam",
    distancia = distancia,
    k = k,
    cluster = unname(resultados_pam[[distancia]][[as.character(k)]]$clustering[ids_artistas])
  )))
}))

medoides_pam <- rbindlist(lapply(names(resultados_pam), function(distancia) {
  rbindlist(lapply(K_CANDIDATOS, function(k) data.table(
    distancia = distancia,
    k = k,
    cluster = seq_len(k),
    artist_id = resultados_pam[[distancia]][[as.character(k)]]$medoids
  )))
}))

## 5. Particiones y tamaños

Las 9 combinaciones en una sola tabla. Los clusters se renumeran por tamaño (1 = el más grande) para leerlos igual en todos los algoritmos. El diagnóstico de tamaños detecta soluciones degeneradas: un cluster que se queda con casi todos los artistas y grupos de unos pocos.

In [ ]:
# ============================================================
# TABLA UNICA DE PARTICIONES, clusters renumerados por tamaño
# ============================================================

particiones <- rbind(particiones_jerarquico, particiones_espectral, particiones_pam)

particiones[
  ,
  cluster := {
    tamanios <- table(cluster)
    orden <- names(sort(tamanios, decreasing = TRUE))
    match(as.character(cluster), orden)
  },
  by = .(algoritmo, distancia, k)
]

# Medoides con la misma numeración que la tabla
medoides_pam <- merge(
  medoides_pam[, .(distancia, k, artist_id)],
  particiones[algoritmo == "pam", .(artist_id, distancia, k, cluster)],
  by = c("artist_id", "distancia", "k")
)
medoides_pam <- merge(medoides_pam, artistas[, .(artist_id = as.character(artist_id), artist_name)], by = "artist_id")

# Sanity check: cada combinación asigna a todos los artistas una sola
# vez y tiene exactamente k clusters
control_particiones <- particiones[
  ,
  .(artistas = .N, unicos = uniqueN(artist_id), clusters = uniqueN(cluster)),
  by = .(algoritmo, distancia, k)
]

stopifnot(
  nrow(control_particiones) == length(ETIQUETAS_ALGORITMO) * length(ETIQUETAS_DISTANCIA) * length(K_CANDIDATOS),
  all(control_particiones$artistas == length(ids_artistas)),
  all(control_particiones$unicos == length(ids_artistas)),
  all(control_particiones$clusters == control_particiones$k)
)

cat("Particiones:", nrow(control_particiones), "(3 algoritmos x 3 distancias x", length(K_CANDIDATOS), "valores de k)\n")

In [ ]:
# ============================================================
# DIAGNOSTICO DE TAMAÑOS
# pct_mayor: % de artistas en el cluster más grande.
# menor: artistas en el cluster más chico.
# ============================================================

tamanios_particiones <- particiones[
  ,
  .N,
  by = .(algoritmo, distancia, k, cluster)
][
  ,
  .(pct_mayor = round(100 * max(N) / sum(N), 1), menor = min(N)),
  by = .(algoritmo, distancia, k)
]

tamanios_particiones[, combinacion := paste(algoritmo, distancia, sep = " | ")]

cat("Porcentaje de artistas en el cluster más grande:\n")
print(dcast(tamanios_particiones, combinacion ~ k, value.var = "pct_mayor"))

cat("\nArtistas en el cluster más chico:\n")
dcast(tamanios_particiones, combinacion ~ k, value.var = "menor")

In [ ]:
# ============================================================
# FIGURA: tamaño del cluster más grande por combinación y k
# ============================================================

datos_figura_tamanios <- copy(tamanios_particiones)
datos_figura_tamanios[
  ,
  `:=`(
    algoritmo_etiqueta = factor(ETIQUETAS_ALGORITMO[algoritmo], levels = ETIQUETAS_ALGORITMO),
    distancia_etiqueta = factor(ETIQUETAS_DISTANCIA[distancia], levels = rev(ETIQUETAS_DISTANCIA))
  )
]

grafico_tamanios <- ggplot(
  datos_figura_tamanios,
  aes(x = factor(k), y = distancia_etiqueta, fill = pct_mayor)
) +
  geom_tile(colour = "white", linewidth = 1) +
  geom_text(aes(label = paste0(round(pct_mayor), "%")), size = 3.6, colour = "#1D1E2C") +
  facet_wrap(~ algoritmo_etiqueta, nrow = 1) +
  scale_fill_gradientn(
    colours = c("#E9F5DB", "#F4A259", "#BC4B51"),
    limits = c(0, 100),
    name = "Artistas en el\ncluster mayor"
  ) +
  labs(
    title = "¿Grupos equilibrados o un cluster dominante?",
    subtitle = "Porcentaje de los 1252 artistas en el cluster más grande, por algoritmo, distancia y k",
    x = "k (cantidad de clusters)",
    y = NULL
  ) +
  theme_minimal(base_size = 13) +
  theme(panel.grid = element_blank(), strip.text = element_text(face = "bold"),
        plot.background = element_rect(fill = "white", colour = NA))

print(grafico_tamanios)

ggsave(
  file.path(FIGURES_DIR, "clustering_03_tamanio_cluster_mayor.png"),
  grafico_tamanios,
  width = 13, height = 4.5, dpi = 300, bg = "white"
)

In [ ]:
# ============================================================
# GUARDADO DE LAS PARTICIONES (archivo local, excluido de git)
# ============================================================

saveRDS(
  list(
    fecha_generacion = Sys.time(),
    archivo_distancias = ARCHIVO_DISTANCIAS,
    configuracion = list(
      k_candidatos = K_CANDIDATOS,
      enlace_jerarquico = ENLACE_JERARQUICO,
      vecino_escala_local = VECINO_ESCALA_LOCAL,
      inicios_kmeans = INICIOS_KMEANS,
      semilla_kmeans_espectral = SEMILLA_KMEANS_ESPECTRAL
    ),
    artistas = artistas,
    particiones = particiones,
    tamanios = tamanios_particiones,
    arboles_jerarquicos = arboles_jerarquicos,
    autovalores_espectral = autovalores_espectral,
    medoides_pam = medoides_pam
  ),
  ARCHIVO_PARTICIONES
)

cat("Particiones guardadas en", ARCHIVO_PARTICIONES, "\n")

## Alcance y limitaciones de la etapa

- Una configuración por algoritmo: enlace de Ward (con DTW y SBD, que no son distancias euclídeas, es una aproximación), una sola escala de afinidad en spectral (7.° vecino) y PAM estándar.
- k de 2 a 8: más grupos no se podrían contar en el video.
- Sin GMM: necesita un espacio euclídeo de variables y una matriz DTW no lo es.
- La elección de k y la estabilidad frente a semillas y remuestreo se evalúan en las etapas 3 y 4.